# Reviewer 2 matched, leakage-safe evaluation

This notebook is a clean evaluation companion to `ML2_COMPLETION (1).ipynb`. It does not overwrite or optimize for the old scores.

Protocol:
- split QTDB records before creating windows (`test_size=0.20`, seed 7);
- split LUDB records before creating windows (10% adaptation, seed 17);
- compute normalization only from QTDB training windows;
- evaluate R1-R5 and adapted R6 on the same untouched LUDB test records;
- report sample metrics, record-level 95% bootstrap intervals, and P/T event-boundary metrics.

Run cells top to bottom. The default path re-evaluates the existing R6 adapted checkpoint; regenerate adaptation separately only after confirming that the fixed adaptation records and eight adaptation epochs are unchanged.

In [1]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt, find_peaks, resample_poly
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score
import wfdb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'PQRST_mapping').exists() and (REPO_ROOT / 'data').exists():
    REPO_ROOT = REPO_ROOT.parent

def first_existing(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return Path(paths[0])

QTDB_DIR = first_existing(
    REPO_ROOT / 'data/qtdb',
    REPO_ROOT / 'PQRST_mapping/data/qtdb',
    REPO_ROOT / 'physionet.org/files/qtdb/1.0.0',
    REPO_ROOT / 'PQRST_mapping/physionet.org/files/qtdb/1.0.0',
    Path('/kaggle/input/datasets/vrishankamembal/qtdb-1-0-0/qt-database-1.0.0'),
)
LUDB_DIR = first_existing(
    REPO_ROOT / 'data/ludb',
    REPO_ROOT / 'PQRST_mapping/data/ludb',
    REPO_ROOT / 'physionet.org/files/ludb/1.0.1/data',
    REPO_ROOT / 'PQRST_mapping/physionet.org/files/ludb/1.0.1/data',
    Path('/kaggle/input/datasets/vrishankamembal/ludb-1-0-1/lobachevsky-university-electrocardiography-database-1.0.1/data'),
)
ARTIFACT_DIR = REPO_ROOT / 'PQRST_mapping/reviewer2_artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)
PRE, POST, R5_POST = 120, 240, 320
BATCH_SIZE = 64
R6_ADAPT_FRACTION, R6_SPLIT_SEED, R6_ADAPT_EPOCHS = 0.10, 17, 8
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINTS = {
    'R1': 'best_ml2_rpeak_guided.pth',
    'R2': 'best_ml2_rpeak_guided_focal.pth',
    'R3': 'best_ml2_rpeak_guided_focal_unweighted.pth',
    'R5': 'best_ml2_r5_post320.pth',
    'R6 QTDB': 'best_ml2_r6_time_qtdb.pth',
    'R6 adapted': 'best_ml2_r6_time_ludb10.pth',
}
print('QTDB:', QTDB_DIR)
print('LUDB:', LUDB_DIR)
print('Device:', DEVICE)

QTDB: /Users/vishwam/VSCode/PQRST_mapping/data/qtdb
LUDB: /Users/vishwam/VSCode/PQRST_mapping/physionet.org/files/ludb/1.0.1/data
Device: mps


## Shared labels, detector, and window builder

In [2]:
def generate_labels(annotation, length, sample_scale=1.0):
    labels = np.zeros(length, dtype=np.int64)
    start, wave_kind = None, None
    for sample, symbol in zip(annotation.sample, annotation.symbol):
        sample = int(round(sample * sample_scale))
        if symbol == '(': start, wave_kind = sample, None
        elif symbol == 'p': wave_kind = 1
        elif symbol == 'N': wave_kind = 2
        elif symbol == 't': wave_kind = 3
        elif symbol == ')' and start is not None and wave_kind is not None:
            labels[max(0, start):min(length, sample + 1)] = wave_kind
            start, wave_kind = None, None
    labels[labels == 2] = 0
    labels[labels == 3] = 2
    return labels

def pan_tompkins_r_peaks(ecg, fs):
    nyquist = fs / 2
    sos = butter(3, [5.0 / nyquist, 18.0 / nyquist], btype='bandpass', output='sos')
    bandpassed = sosfiltfilt(sos, ecg)
    derivative = np.convolve(bandpassed, np.array([-1, -2, 0, 2, 1]) * fs / 8, mode='same')
    width = max(1, round(0.150 * fs))
    integrated = np.convolve(derivative ** 2, np.ones(width) / width, mode='same')
    candidates, _ = find_peaks(integrated, distance=max(1, round(0.20 * fs)))
    boot = candidates[candidates < min(len(ecg), round(2 * fs))]
    spki = np.percentile(integrated[boot], 90) if len(boot) else 0.0
    npki = np.percentile(integrated[boot], 25) if len(boot) else 0.0
    accepted = []
    for peak in candidates:
        threshold = npki + 0.25 * (spki - npki)
        if integrated[peak] >= threshold:
            accepted.append(peak); spki = 0.125 * integrated[peak] + 0.875 * spki
        else:
            npki = 0.125 * integrated[peak] + 0.875 * npki
    search = round(0.10 * fs)
    refined = []
    for peak in accepted:
        left, right = max(0, peak - search), min(len(ecg), peak + search + 1)
        refined.append(left + np.argmax(ecg[left:right]))
    return np.unique(np.asarray(refined, dtype=int))

def create_windows(ecg, labels, r_peaks, post):
    x, y = [], []
    for r in r_peaks:
        left, right = r - PRE, r + post
        if left >= 0 and right <= len(ecg):
            x.append(ecg[left:right]); y.append(labels[left:right])
    return np.asarray(x, dtype=np.float32), np.asarray(y, dtype=np.int64)

def qtdb_record(record_name, post):
    path = str(QTDB_DIR / record_name)
    record = wfdb.rdrecord(path)
    ecg = record.p_signal[:, 0].astype(np.float32)
    labels = generate_labels(wfdb.rdann(path, 'pu0'), len(ecg))
    return create_windows(ecg, labels, pan_tompkins_r_peaks(ecg, float(record.fs)), post)

def ludb_record(record_name, post):
    path = str(LUDB_DIR / record_name)
    record = wfdb.rdrecord(path)
    lead = record.sig_name.index('ii')
    ecg = resample_poly(record.p_signal[:, lead].astype(np.float32), up=1, down=2)
    labels = generate_labels(wfdb.rdann(path, 'ii'), len(ecg), sample_scale=0.5)
    size = min(len(ecg), len(labels))
    ecg, labels = ecg[:size], labels[:size]
    return create_windows(ecg, labels, pan_tompkins_r_peaks(ecg, 250.0), post)

def build_partition(record_names, builder, post):
    xs, ys, ids, skipped = [], [], [], []
    for record_name in record_names:
        try:
            x, y = builder(record_name, post)
            if len(x):
                xs.append(x); ys.append(y); ids.extend([record_name] * len(x))
        except Exception as error:
            skipped.append({'record': record_name, 'error': str(error)})
    if not xs:
        raise RuntimeError('No usable windows were generated')
    return np.concatenate(xs), np.concatenate(ys), np.asarray(ids), skipped

## Record-level partitions and train-only normalization

In [3]:
qtdb_records = sorted(path.stem for path in QTDB_DIR.glob('*.hea'))
if not qtdb_records: qtdb_records = sorted(path.stem for path in QTDB_DIR.glob('*.dat'))
qtdb_train_records, qtdb_val_records = train_test_split(qtdb_records, test_size=0.20, random_state=7, shuffle=True)
qtdb_train_records, qtdb_val_records = sorted(qtdb_train_records), sorted(qtdb_val_records)
assert set(qtdb_train_records).isdisjoint(qtdb_val_records)
assert set(qtdb_train_records) | set(qtdb_val_records) == set(qtdb_records)

# Windows are created independently after the record split.
X_qt_train, Y_qt_train, qt_train_ids, qt_train_skipped = build_partition(qtdb_train_records, qtdb_record, POST)
X_qt_val, Y_qt_val, qt_val_ids, qt_val_skipped = build_partition(qtdb_val_records, qtdb_record, POST)
train_mean = float(X_qt_train.mean())
train_std = float(X_qt_train.std())
if train_std == 0: raise ValueError('QTDB training standard deviation is zero')
X_qt_train = (X_qt_train - train_mean) / train_std
X_qt_val = (X_qt_val - train_mean) / train_std

ludb_records = sorted(path.stem for path in LUDB_DIR.glob('*.hea'))
r6_adapt_records, r6_test_records = train_test_split(ludb_records, test_size=1 - R6_ADAPT_FRACTION, random_state=R6_SPLIT_SEED, shuffle=True)
r6_adapt_records, r6_test_records = sorted(r6_adapt_records), sorted(r6_test_records)
assert set(r6_adapt_records).isdisjoint(r6_test_records)
assert set(r6_adapt_records) | set(r6_test_records) == set(ludb_records)

# Build adaptation and untouched test windows separately, for each model window length.
X_lu_adapt_240, Y_lu_adapt_240, lu_adapt_ids_240, _ = build_partition(r6_adapt_records, ludb_record, POST)
X_lu_test_240, Y_lu_test_240, lu_test_ids_240, _ = build_partition(r6_test_records, ludb_record, POST)
X_lu_adapt_320, Y_lu_adapt_320, lu_adapt_ids_320, _ = build_partition(r6_adapt_records, ludb_record, R5_POST)
X_lu_test_320, Y_lu_test_320, lu_test_ids_320, _ = build_partition(r6_test_records, ludb_record, R5_POST)

for ids in [qt_train_ids, qt_val_ids, lu_adapt_ids_240, lu_test_ids_240, lu_adapt_ids_320, lu_test_ids_320]:
    assert len(ids) == len(set(ids)) or len(ids) > 0
assert set(qt_train_ids).isdisjoint(qt_val_ids)
assert set(lu_adapt_ids_240).isdisjoint(lu_test_ids_240)
assert set(lu_adapt_ids_320).isdisjoint(lu_test_ids_320)

split_manifest = {
    'qtdb_train_records': qtdb_train_records, 'qtdb_validation_records': qtdb_val_records,
    'ludb_adaptation_records': r6_adapt_records, 'ludb_untouched_test_records': r6_test_records,
    'qtdb_train_mean': train_mean, 'qtdb_train_std': train_std,
}
(ARTIFACT_DIR / 'reviewer2_split_manifest.json').write_text(json.dumps(split_manifest, indent=2))
np.savez(ARTIFACT_DIR / 'reviewer2_qtdb_normalization.npz', mean=train_mean, std=train_std)
print('QTDB records:', len(qtdb_train_records), 'train /', len(qtdb_val_records), 'validation')
print('LUDB records:', len(r6_adapt_records), 'adaptation /', len(r6_test_records), 'untouched test')
print('Windows:', len(X_qt_train), len(X_qt_val), len(X_lu_test_240), len(X_lu_test_320))
print('Normalization: mean=', train_mean, 'std=', train_std)

QTDB records: 5 train / 2 validation
LUDB records: 20 adaptation / 180 untouched test
Windows: 5108 1800 1710 1655
Normalization: mean= 1.8708250522613525 std= 2.6616458892822266


## Models, locked R4 decoder, and checkpoint scoring

In [4]:
class CNNFeatureExtractor(nn.Module):
    def __init__(self, channels=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(channels, 32, kernel_size=7, padding=3), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
        )

    def forward(self, x):
        return self.features(x)

class BiLSTMBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(64, 128, num_layers=1, batch_first=True, bidirectional=True)

    def forward(self, x):
        return self.lstm(x)[0]

class RPeakGuidedML2(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.cnn = CNNFeatureExtractor()
        self.bilstm = BiLSTMBlock()
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)
        return self.classifier(self.dropout(self.bilstm(x)))

class RPeakTimeML2(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(2, 32, kernel_size=7, padding=3), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.bilstm = nn.LSTM(64, 128, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)
        return self.classifier(self.dropout(self.bilstm(x)[0]))

def time_channel(signals, post):
    time = (np.arange(signals.shape[1], dtype=np.float32) - PRE) / post
    return np.stack([signals.astype(np.float32), np.broadcast_to(time, signals.shape)], axis=1).copy()

def predict(model, features):
    model.eval(); predictions = []
    with torch.inference_mode():
        for start in range(0, len(features), BATCH_SIZE):
            batch = torch.as_tensor(features[start:start + BATCH_SIZE], dtype=torch.float32, device=DEVICE)
            predictions.append(model(batch).argmax(2).cpu().numpy())
    return np.concatenate(predictions)

def predict_probabilities(model, features):
    model.eval(); probabilities = []
    with torch.inference_mode():
        for start in range(0, len(features), BATCH_SIZE):
            batch = torch.as_tensor(features[start:start + BATCH_SIZE], dtype=torch.float32, device=DEVICE)
            probabilities.append(torch.softmax(model(batch), dim=2).cpu().numpy())
    return np.concatenate(probabilities)

R4_T_START, R4_T_MIN_LENGTH = PRE + 5, 4
def r4_decode(probabilities, threshold):
    decoded = probabilities.argmax(axis=2).astype(np.int64)
    for beat, probs in enumerate(probabilities):
        decoded[beat, PRE:] = np.where(decoded[beat, PRE:] == 1, 0, decoded[beat, PRE:])
        decoded[beat, decoded[beat] == 2] = 0
        active = np.zeros(len(probs), dtype=bool); active[R4_T_START:] = probs[R4_T_START:, 2] >= threshold
        edges = np.diff(np.r_[False, active, False].astype(int))
        choices = [(a, b - 1) for a, b in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)) if b - a >= R4_T_MIN_LENGTH]
        if choices:
            a, b = max(choices, key=lambda part: probs[part[0]:part[1] + 1, 2].sum()); decoded[beat, a:b + 1] = 2
    return decoded

def classification_row(name, truth, prediction, records, post):
    report = classification_report(truth.ravel(), prediction.ravel(), labels=[0, 1, 2], target_names=['Background', 'P Wave', 'T Wave'], output_dict=True, zero_division=0)
    return {'Experiment': name, 'Records': len(np.unique(records)), 'Windows': len(truth), 'Post samples': post, 'Accuracy': accuracy_score(truth.ravel(), prediction.ravel()), 'P precision': report['P Wave']['precision'], 'P recall': report['P Wave']['recall'], 'P F1': report['P Wave']['f1-score'], 'T precision': report['T Wave']['precision'], 'T recall': report['T Wave']['recall'], 'T F1': report['T Wave']['f1-score'], 'Macro F1': report['macro avg']['f1-score'], 'Weighted F1': report['weighted avg']['f1-score']}

def load_checkpoint(path, time=False):
    model = (RPeakTimeML2() if time else RPeakGuidedML2()).to(DEVICE)
    checkpoint = Path(path)
    if not checkpoint.is_absolute(): checkpoint = Path.cwd() / checkpoint
    if not checkpoint.exists(): checkpoint = Path(REPO_ROOT) / 'PQRST_mapping' / path
    if not checkpoint.exists(): raise FileNotFoundError(checkpoint)
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))
    return model

# R4 threshold is selected on QTDB validation only. LUDB labels are not read in this cell.
r4_base = load_checkpoint(CHECKPOINTS['R1'])
r4_val_probs = predict_probabilities(r4_base, X_qt_val[:, None])
threshold_rows = []
for threshold in np.arange(0.10, 0.76, 0.05):
    pred = r4_decode(r4_val_probs, threshold)
    threshold_rows.append((threshold, f1_score(Y_qt_val.ravel(), pred.ravel(), labels=[0, 1, 2], average='macro', zero_division=0)))
R4_T_THRESHOLD = float(max(threshold_rows, key=lambda row: row[1])[0])
print('R4 threshold locked from QTDB validation:', R4_T_THRESHOLD)

R4 threshold locked from QTDB validation: 0.7500000000000002


In [5]:
# Matched evaluation: every row below uses the same r6_test_records.
X240_test = (X_lu_test_240 - train_mean) / train_std
X320_test = (X_lu_test_320 - train_mean) / train_std
results, predictions_by_experiment = [], {}

def score(name, model, features, truth, ids, post, decoder=None):
    probabilities = predict_probabilities(model, features)
    prediction = decoder(probabilities) if decoder else probabilities.argmax(2)
    predictions_by_experiment[name] = (truth, prediction, ids, post)
    results.append(classification_row(name, truth, prediction, ids, post))

for name in ['R1', 'R2', 'R3']:
    model = load_checkpoint(CHECKPOINTS[name])
    score(name, model, X240_test[:, None], Y_lu_test_240, lu_test_ids_240, POST)
score('R4', r4_base, X240_test[:, None], Y_lu_test_240, lu_test_ids_240, POST, lambda p: r4_decode(p, R4_T_THRESHOLD))
r5_model = load_checkpoint(CHECKPOINTS['R5'])
score('R5', r5_model, X320_test[:, None], Y_lu_test_320, lu_test_ids_320, R5_POST)
r6_model = load_checkpoint(CHECKPOINTS['R6 adapted'], time=True)
score('R6 adapted', r6_model, time_channel(X320_test, R5_POST), Y_lu_test_320, lu_test_ids_320, R5_POST)
matched_metrics = pd.DataFrame(results)
display(matched_metrics.round(4))
matched_metrics.to_csv(ARTIFACT_DIR / 'matched_ludb_test_metrics.csv', index=False)

,Experiment,Records,Windows,Post samples,Accuracy,P precision,P recall,P F1,T precision,T recall,T F1,Macro F1,Weighted F1
0,R1,180,1710,240,0.8780,0.7785,0.8085,0.7932,0.7346,0.7415,0.7381,0.8166,0.8784
1,R2,180,1710,240,0.8161,0.6071,0.9195,0.7313,0.5867,0.7655,0.6643,0.7551,0.8246
2,R3,180,1710,240,0.8637,0.7783,0.7379,0.7576,0.7071,0.7051,0.7061,0.7911,0.8633
3,R4,180,1710,240,0.8377,0.7735,0.4323,0.5547,0.8114,0.5364,0.6459,0.6995,0.8230
4,R5,180,1655,320,0.8641,0.7740,0.7660,0.7700,0.7323,0.7420,0.7371,0.8050,0.8642
5,R6 adapted,180,1655,320,0.8726,0.7565,0.8291,0.7911,0.7427,0.7916,0.7663,0.8230,0.8740


## Record-level confidence intervals and event-boundary analysis

In [6]:
def intervals(labels, wave_class):
    mask = labels == wave_class
    edges = np.diff(np.r_[False, mask, False].astype(int))
    return list(zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1) - 1))

def event_metrics(truth, prediction, ids, tolerance_ms=100.0, fs=250.0):
    rows = []
    for record in np.unique(ids):
        record_mask = ids == record
        for wave_class, wave_name in [(1, 'P'), (2, 'T')]:
            true_events = [event for beat in truth[record_mask] for event in intervals(beat, wave_class)]
            pred_events = [event for beat in prediction[record_mask] for event in intervals(beat, wave_class)]
            used = set(); onset_errors, offset_errors = [], []
            for start, end in true_events:
                candidates = [(max(0, min(end, ps) - max(start, pe) + 1), i, ps, pe) for i, (ps, pe) in enumerate(pred_events) if i not in used]
                candidates = [item for item in candidates if item[0] > 0]
                if not candidates: continue
                _, index, ps, pe = max(candidates)
                used.add(index); onset_errors.append((ps - start) / fs * 1000); offset_errors.append((pe - end) / fs * 1000)
            matched = len(onset_errors)
            rows.append({'record': record, 'Wave': wave_name, 'True events': len(true_events), 'Predicted events': len(pred_events), 'Matched events': matched, 'Sensitivity': matched / len(true_events) if true_events else np.nan, 'PPV': matched / len(pred_events) if pred_events else np.nan, 'Missed': len(true_events) - matched, 'False positives': len(pred_events) - matched, 'Onset bias ms': np.mean(onset_errors) if onset_errors else np.nan, 'Onset SD ms': np.std(onset_errors, ddof=1) if len(onset_errors) > 1 else np.nan, 'Offset bias ms': np.mean(offset_errors) if offset_errors else np.nan, 'Offset SD ms': np.std(offset_errors, ddof=1) if len(offset_errors) > 1 else np.nan})
    return pd.DataFrame(rows)

def record_metric_table(truth, prediction, ids):
    rows = []
    for record in np.unique(ids):
        mask = ids == record
        rows.append({'record': record, 'Macro F1': f1_score(truth[mask].ravel(), prediction[mask].ravel(), labels=[0, 1, 2], average='macro', zero_division=0), 'P F1': f1_score(truth[mask].ravel(), prediction[mask].ravel(), labels=[1], average='macro', zero_division=0), 'T F1': f1_score(truth[mask].ravel(), prediction[mask].ravel(), labels=[2], average='macro', zero_division=0)})
    return pd.DataFrame(rows)

def bootstrap_ci(values, repetitions=2000, seed=2026):
    values = np.asarray(values, dtype=float); values = values[np.isfinite(values)]
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(repetitions, len(values)), replace=True).mean(axis=1)
    return float(np.mean(values)), float(np.percentile(samples, 2.5)), float(np.percentile(samples, 97.5))

ci_rows, event_rows = [], []
for name, (truth, prediction, ids, post) in predictions_by_experiment.items():
    per_record = record_metric_table(truth, prediction, ids)
    for metric in ['Macro F1', 'P F1', 'T F1']:
        mean, low, high = bootstrap_ci(per_record[metric])
        ci_rows.append({'Experiment': name, 'Metric': metric, 'Record mean': mean, '95% CI low': low, '95% CI high': high, 'Records': len(per_record)})
    event_table = event_metrics(truth, prediction, ids)
    event_summary = event_table.groupby('Wave')[['Sensitivity', 'PPV', 'Missed', 'False positives', 'Onset bias ms', 'Onset SD ms', 'Offset bias ms', 'Offset SD ms']].mean().reset_index()
    event_summary.insert(0, 'Experiment', name); event_rows.append(event_summary)

confidence_intervals = pd.DataFrame(ci_rows)
boundary_metrics = pd.concat(event_rows, ignore_index=True)
display(confidence_intervals.round(4))
display(boundary_metrics.round(4))
confidence_intervals.to_csv(ARTIFACT_DIR / 'record_bootstrap_confidence_intervals.csv', index=False)
boundary_metrics.to_csv(ARTIFACT_DIR / 'event_boundary_metrics.csv', index=False)
print('QRS event metrics are not reported: the current 3-class target mapping intentionally collapses QRS into background.')

,Experiment,Metric,Record mean,95% CI low,95% CI high,Records
0,R1,Macro F1,0.7850,0.7629,0.8055,180
1,R1,P F1,0.7005,0.6565,0.7400,180
2,R1,T F1,0.7340,0.7054,0.7614,180
3,R2,Macro F1,0.7470,0.7253,0.7682,180
4,R2,P F1,0.6974,0.6572,0.7342,180
5,R2,T F1,0.6741,0.6485,0.7002,180
6,R3,Macro F1,0.7563,0.7323,0.7786,180
7,R3,P F1,0.6550,0.6092,0.6963,180
8,R3,T F1,0.7014,0.6691,0.7314,180
9,R4,Macro F1,0.6844,0.6640,0.7039,180


,Experiment,Wave,Sensitivity,PPV,Missed,False positives,Onset bias ms,Onset SD ms,Offset bias ms,Offset SD ms
0,R1,P,0.0074,0.0053,13.6389,18.5389,50.1429,19.7990,-33.0000,38.6552
1,R1,T,0.0185,0.0112,12.8500,18.9500,64.8333,57.1045,-58.0741,33.2205
2,R2,P,0.0180,0.0098,13.4778,24.8611,70.6458,18.4193,-32.9375,18.1012
3,R2,T,0.0253,0.0155,12.6889,22.1167,33.0000,32.3290,-64.0071,26.5249
4,R3,P,0.0118,0.0077,13.5889,17.3222,58.8704,46.2852,-42.8889,49.8025
5,R3,T,0.0240,0.0158,12.7389,16.6389,56.1250,37.9627,-46.0350,20.5988
6,R4,P,0.0019,0.0030,13.7056,9.3833,52.8000,NaN,-57.6000,NaN
7,R4,T,0.0000,0.0000,13.1333,8.9556,NaN,NaN,NaN,NaN
8,R5,P,0.0086,0.0066,14.6611,19.0889,75.4917,17.5621,-27.5083,19.9595
9,R5,T,0.0182,0.0109,17.2889,24.2889,64.7444,50.1287,-65.3365,59.1418


QRS event metrics are not reported: the current 3-class target mapping intentionally collapses QRS into background.


## Leakage and decision audit

The checkpoint re-evaluation above does not fit on LUDB. R6's adapted checkpoint is accepted only as an existing artifact; if it is regenerated, adaptation must use `r6_adapt_records` only, for the fixed `R6_ADAPT_EPOCHS`, and the untouched test arrays must not be created until adaptation is complete.

The current target mapping collapses QRS to background, so a separate QRS boundary result requires a new target definition and retraining; this notebook does not relabel the existing models to manufacture that metric.